# Phase 3 — rerun of all experiments on corrected labels

Paper: *Leakage-Audited Solar Flare Forecasting Across Solar Cycles 24 and 25* (Avinash Kumar, ORCID 0009-0008-7613-1185).

Phase 2 found that the Cycle 24 labels (SWAN-SF flare list) carry the SWPC 0.7 scaling of GOES XRS-B, while the Cycle 25 labels (GOES-16 science data) are in true flux. This notebook reruns every experiment of v1 with one label column at a time:

- `v1` — the published labels (used to check that this code reproduces v1);
- `v2c` — both cycles from the NOAA GOES Flare Report (science-quality, true flux), region numbers from the report, else SWAN-SF, else HEK.

Changes from v1 besides the labels (from the Phase 1–2 review): 1,000 bootstrap resamples; the logistic-regression grid extended to C = 0.0003 and 0.001; paired tests for D − A (Cycle 25 → 24 asymmetry) in addition to B − C; a size-matched Cycle 24 training set; Brier skill against both the Cycle 24 and the early Cycle 25 climatology; every number written to CSV and every figure drawn from CSV.

Input: `c24_samples_v2.pkl` and `c25_samples_v2.pkl` (Cycle 25 to 30 Dec 2024) or `c25ext_samples_v2.pkl` (to 30 Dec 2025) from the audit notebook. Environment variables: `LABEL` (default `v2c`), `C25_TABLE` (default `c25_samples_v2.pkl`), `RUN_NAME` (default = `LABEL`). Output: `phase3_out/<RUN_NAME>/`.

In [1]:
import os, json, time, itertools, platform, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss

WORK = os.environ.get("AUDIT_WORK", "/content"); os.chdir(WORK)
LABEL = os.environ.get("LABEL", "v2c")
PUBLISHED = os.environ.get("PUBLISHED_DIR", "")      # folder with the v1 result CSVs, for the reproduction check
NB = int(os.environ.get("NBOOT", "1000"))
C25_TABLE = os.environ.get("C25_TABLE", "c25_samples_v2.pkl")
RUN = os.environ.get("RUN_NAME", LABEL)
OUT = f"phase3_out/{RUN}"; os.makedirs(OUT, exist_ok=True)
T0 = time.time()

def save(df, name, note=""):
    df.to_csv(f"{OUT}/{name}.csv", index=False)
    print(f"\n--> {OUT}/{name}.csv" + (f"  ({note})" if note else "") + f"   [{round((time.time()-T0)/60, 1)} min]")
    print(df.to_string(index=False), flush=True); return df

A = pd.read_pickle("c24_samples_v2.pkl").reset_index(drop=True)
B = pd.read_pickle(C25_TABLE).reset_index(drop=True)
YEARS = list(range(2021, int(B.time.dt.year.max()) + 1))
FE = [x for x in A.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]
assert len(FE) == 68, len(FE)
A["y"] = A["y_" + LABEL].astype(int); B["y"] = B["y_" + LABEL].astype(int)
print("run", RUN, "| table", C25_TABLE, "| C25 windows", B.time.min(), "to", B.time.max(), "| years", YEARS)
print("label", LABEL, "| C24", A.shape, "pos", round(A.y.mean(), 4), "| C25", B.shape, "pos", round(B.y.mean(), 4))
print({"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__})

run v1 | table c25_samples_v2.pkl | C25 windows 2019-03-05 19:00:00 to 2024-12-30 00:00:00 | years [2021, 2022, 2023, 2024]
label v1 | C24 (215463, 77) pos 0.0371 | C25 (162801, 77) pos 0.0816
{'python': '3.12.15', 'sklearn': '1.9.1', 'pandas': '3.1.0', 'numpy': '2.5.4'}


## Helpers (same definitions as v1)

In [2]:
def mk(name, **kw):
    """Default models of v1; keyword arguments override (tuned settings)."""
    if name in ("1-feat (USFLUX)", "3-feat (USFLUX,TOTUSJH,R_VALUE)"):
        col = {c: i for i, c in enumerate(FE)}
        cols = [col["USFLUX_last"]] if name.startswith("1") else [col[c] for c in ["USFLUX_last", "TOTUSJH_last", "R_VALUE_last"]]
        ct = ColumnTransformer([("s", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), cols)])
        return make_pipeline(ct, LogisticRegression(class_weight="balanced", max_iter=300))
    if name == "LR": return make_pipeline(StandardScaler(), LogisticRegression(**{"C": 0.1, "class_weight": "balanced", "max_iter": 300, **kw}))
    if name == "HGB": return HistGradientBoostingClassifier(**{"max_iter": 150, "learning_rate": 0.08, "class_weight": "balanced", "random_state": 0, **kw})
    return make_pipeline(StandardScaler(), MLPClassifier(**{"hidden_layer_sizes": (64, 32), "early_stopping": True, "max_iter": 80, "random_state": 0, **kw}))

def tss_pt(p, yy, t):
    pr = p >= t; tp = (pr & (yy == 1)).sum(); fn = ((~pr) & (yy == 1)).sum(); fp = (pr & (yy == 0)).sum(); tn = ((~pr) & (yy == 0)).sum()
    return tp / max(tp + fn, 1) - fp / max(fp + tn, 1)
def best_t(p, yy):
    ts = np.unique(np.quantile(p, np.linspace(.5, .999, 120))); return max(ts, key=lambda t: tss_pt(p, yy, t))
def counts(p, yy, codes, n, t):
    """Per-region confusion counts [TP, FN, FP, TN]."""
    pr = p >= t; f = lambda w: np.bincount(codes, weights=w.astype(float), minlength=n)
    return np.c_[f(pr & (yy == 1)), f((~pr) & (yy == 1)), f(pr & (yy == 0)), f((~pr) & (yy == 0))]
def tss_c(c): s = c.sum(0); return s[0] / max(s[0] + s[1], 1) - s[2] / max(s[2] + s[3], 1)
def hss_c(c):
    tp, fn, fp, tn = c.sum(0); d = (tp + fn) * (fn + tn) + (tp + fp) * (fp + tn); return 2 * (tp * tn - fn * fp) / d if d else 0.0
def boot_idx(n, seed=1): return np.random.RandomState(seed).randint(0, n, (NB, n))
def ci(vals): return f"[{np.percentile(vals, 2.5):.3f}, {np.percentile(vals, 97.5):.3f}]"
def boot(c, seed=1):
    c = c[c.sum(1) > 0]; idx = boot_idx(len(c), seed); return np.array([tss_c(c[i]) for i in idx])
def boot_paired(c1, c2, seed=2):
    """TSS(c1) - TSS(c2) with the same region resamples; rows must be the same regions in the same order."""
    keep = (c1.sum(1) > 0) & (c2.sum(1) > 0); c1, c2 = c1[keep], c2[keep]; idx = boot_idx(len(c1), seed)
    return tss_c(c1) - tss_c(c2), np.array([tss_c(c1[i]) - tss_c(c2[i]) for i in idx])
def val_split(D, seed=0):
    ars = D.ar.unique(); v = set(np.random.RandomState(seed).choice(ars, int(.2 * len(ars)), replace=False))
    return D[~D.ar.isin(v)], D[D.ar.isin(v)]

X, y = A[FE].values, A.y.values
codes, uars = pd.factorize(A.ar); NA = len(uars)
MODELS = ["1-feat (USFLUX)", "3-feat (USFLUX,TOTUSJH,R_VALUE)", "LR", "HGB", "MLP"]
SHORT = {"1-feat (USFLUX)": "1-feature", "3-feat (USFLUX,TOTUSJH,R_VALUE)": "3-feature", "LR": "LR", "HGB": "HGB", "MLP": "MLP"}

## E1 — Leakage audit, default hyperparameters (Cycle 24; v1 Section 4.1, Table 2)

In [3]:
rng = np.random.RandomState(0)
splits = {"random rows (leaky)": list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y)),
          "grouped by AR": list(GroupKFold(5).split(X, y, codes))}
tr0 = np.where(A.time < "2014-01-01")[0]
te0 = np.where((A.time >= "2014-01-15") & (~A.ar.isin(set(A.ar.iloc[tr0]))))[0]
splits["forward time + AR-disjoint"] = [(tr0, te0)]
res, CNT1 = [], {}
for mn in MODELS:
    for sn, folds in splits.items():
        C = np.zeros((NA, 4))
        for a, te in folds:
            if sn.startswith("random"):
                v = rng.rand(len(a)) < .2; tr, va = a[~v], a[v]
            else:
                ua = np.unique(codes[a]); vs = set(rng.choice(ua, int(.2 * len(ua)), replace=False))
                isv = np.isin(codes[a], list(vs)); tr, va = a[~isv], a[isv]
            m = mk(mn).fit(X[tr], y[tr]); t = best_t(m.predict_proba(X[va])[:, 1], y[va])
            C += counts(m.predict_proba(X[te])[:, 1], y[te], codes[te], NA, t)
        CNT1[(mn, sn)] = C; b = boot(C)
        res.append(dict(model=mn, split=sn, TSS=round(tss_c(C), 3), CI95=ci(b), HSS2=round(hss_c(C), 3)))
    d, bd = boot_paired(CNT1[(mn, "random rows (leaky)")], CNT1[(mn, "grouped by AR")])
    res.append(dict(model=mn, split="inflation random - grouped (paired)", TSS=round(d, 3), CI95=ci(bd), HSS2=np.nan))
    print(mn, "done", round((time.time() - T0) / 60, 1), "min", flush=True)
E1 = save(pd.DataFrame(res), "e1_leakage_default")

1-feat (USFLUX) done 0.0 min


3-feat (USFLUX,TOTUSJH,R_VALUE) done 0.1 min


LR done 0.7 min


HGB done 5.2 min


MLP done 10.8 min



--> phase3_out/v1/e1_leakage_default.csv   [10.8 min]
                          model                               split   TSS            CI95  HSS2
                1-feat (USFLUX)                 random rows (leaky) 0.636  [0.593, 0.672] 0.172
                1-feat (USFLUX)                       grouped by AR 0.633  [0.591, 0.666] 0.166
                1-feat (USFLUX)          forward time + AR-disjoint 0.647  [0.593, 0.694] 0.160
                1-feat (USFLUX) inflation random - grouped (paired) 0.003 [-0.011, 0.018]   NaN
3-feat (USFLUX,TOTUSJH,R_VALUE)                 random rows (leaky) 0.683  [0.645, 0.717] 0.219
3-feat (USFLUX,TOTUSJH,R_VALUE)                       grouped by AR 0.654  [0.605, 0.697] 0.256
3-feat (USFLUX,TOTUSJH,R_VALUE)          forward time + AR-disjoint 0.690  [0.612, 0.746] 0.283
3-feat (USFLUX,TOTUSJH,R_VALUE) inflation random - grouped (paired) 0.030  [0.006, 0.061]   NaN
                             LR                 random rows (leaky) 0.747  [0.711

## E2 — Hyperparameter tuning (grouped 3-fold CV on 2010–2013; logistic-regression grid extended)

In [4]:
GRID = {
 "LR":  [dict(C=c) for c in (0.0003, 0.001, 0.003, 0.03, 0.3)],
 "HGB": [dict(learning_rate=0.06, max_iter=i, max_leaf_nodes=l, min_samples_leaf=s, l2_regularization=r) for i, l, s, r in
         [(100, 8, 200, 1), (300, 8, 200, 10), (100, 31, 500, 10), (300, 31, 500, 10), (150, 15, 100, 1), (150, 15, 500, 10), (300, 8, 500, 1), (100, 31, 50, 1)]],
 "MLP": [dict(hidden_layer_sizes=h, alpha=a) for h, a in [((32,), 1e-3), ((32,), 1e-1), ((32,), 1.0), ((32,), 10.0), ((64, 32), 1.0)]],
}
tr_all = A[A.time < "2014-01-01"].reset_index(drop=True); tune = tr_all.iloc[::3]
g = pd.factorize(tune.ar)[0]; Xt, yt = tune[FE].values, tune.y.values
BEST, rows = {}, []
for mn, grid in GRID.items():
    for kw in grid:
        aucs = [roc_auc_score(yt[v], mk(mn, **kw).fit(Xt[a], yt[a]).predict_proba(Xt[v])[:, 1]) for a, v in GroupKFold(3).split(Xt, yt, g)]
        rows.append(dict(model=mn, params=json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in kw.items()}), grouped_cv_auc=round(np.mean(aucs), 4)))
    sub = [r for r in rows if r["model"] == mn]; i = int(np.argmax([r["grouped_cv_auc"] for r in sub])); BEST[mn] = grid[i]
    for j, r in enumerate(sub): r["selected"] = (j == i); r["at_grid_edge"] = (j == i) and (j in (0, len(grid) - 1))
E2 = save(pd.DataFrame(rows), "e2_tuning_grid")
print("selected:", BEST)


--> phase3_out/v1/e2_tuning_grid.csv   [15.4 min]
model                                                                                                           params  grouped_cv_auc  selected  at_grid_edge
   LR                                                                                                    {"C": 0.0003}          0.9289     False         False
   LR                                                                                                     {"C": 0.001}          0.9290      True         False
   LR                                                                                                     {"C": 0.003}          0.9284     False         False
   LR                                                                                                      {"C": 0.03}          0.9258     False         False
   LR                                                                                                       {"C": 0.3}          0.9233     False         F

selected: {'LR': {'C': 0.001}, 'HGB': {'learning_rate': 0.06, 'max_iter': 100, 'max_leaf_nodes': 31, 'min_samples_leaf': 500, 'l2_regularization': 10}, 'MLP': {'hidden_layer_sizes': (64, 32), 'alpha': 1.0}}


## E3 — Leakage audit, tuned models, 5-fold, paired inflation (v1 Table 3)

In [5]:
rng = np.random.RandomState(0); out = []
for mn in ["LR", "HGB", "MLP"]:
    Cs = {}
    for sn, fl in [("random", splits["random rows (leaky)"]), ("grouped", splits["grouped by AR"])]:
        C = np.zeros((NA, 4))
        for a, te in fl:
            if sn == "random": v = rng.rand(len(a)) < .2
            else:
                ua = np.unique(codes[a]); v = np.isin(codes[a], rng.choice(ua, int(.2 * len(ua)), replace=False))
            m = mk(mn, **BEST[mn]).fit(X[a[~v]], y[a[~v]]); t = best_t(m.predict_proba(X[a[v]])[:, 1], y[a[v]])
            C += counts(m.predict_proba(X[te])[:, 1], y[te], codes[te], NA, t)
        Cs[sn] = C
    d, bd = boot_paired(Cs["random"], Cs["grouped"], seed=1)
    out.append(dict(model=mn + " (tuned)", TSS_random=round(tss_c(Cs["random"]), 3), CI_r=ci(boot(Cs["random"])),
                    TSS_grouped=round(tss_c(Cs["grouped"]), 3), CI_g=ci(boot(Cs["grouped"])),
                    inflation=round(d, 3), inflation_CI=ci(bd)))
E3 = save(pd.DataFrame(out), "e3_leakage_tuned")


--> phase3_out/v1/e3_leakage_tuned.csv   [21.1 min]
      model  TSS_random           CI_r  TSS_grouped           CI_g  inflation   inflation_CI
 LR (tuned)       0.743 [0.707, 0.772]        0.726 [0.690, 0.754]      0.018 [0.009, 0.026]
HGB (tuned)       0.860 [0.845, 0.874]        0.703 [0.660, 0.736]      0.157 [0.130, 0.192]
MLP (tuned)       0.737 [0.701, 0.766]        0.724 [0.688, 0.755]      0.013 [0.003, 0.022]


## E4 — Cross-cycle transfer A–D, paired B − C and D − A, size-matched Cycle 24 training (v1 Section 4.3)

In [6]:
a_tr0 = A[A.time < "2014-01-01"]; a_te = A[(A.time >= "2014-01-15") & (~A.ar.isin(set(a_tr0.ar)))]
b_tr0 = B[B.time < "2023-01-01"]; b_te = B[(B.time >= "2023-01-15") & (~B.ar.isin(set(b_tr0.ar)))]
a_tr, a_va = val_split(a_tr0); b_tr, b_va = val_split(b_tr0)
print("C24 train/val/test", len(a_tr), len(a_va), len(a_te), "| C25 train/val/test", len(b_tr), len(b_va), len(b_te))
ca, na = pd.factorize(a_te.ar); ca_n = len(na)
cb, nb = pd.factorize(b_te.ar); cb_n = len(nb)
TEST = {"C24": (a_te, ca, ca_n), "C25": (b_te, cb, cb_n)}

def run(model, kw, tr, va, test_key):
    te, cc, n = TEST[test_key]
    m = mk(model, **kw).fit(tr[FE].values, tr.y.values); t = best_t(m.predict_proba(va[FE].values)[:, 1], va.y.values)
    p = m.predict_proba(te[FE].values)[:, 1]
    return counts(p, te.y.values, cc, n, t), roc_auc_score(te.y.values, p), m, t

SETUPS = [("A", "in-cycle C24 (2010-13 -> 2014-18)", "C24", "a"), ("B", f"cross C24 -> C25 (2023-{YEARS[-1] % 100})", "C25", "a"),
          ("C", f"in-cycle C25 (2021-22 -> 2023-{YEARS[-1] % 100})", "C25", "b"), ("D", "reverse C25 -> C24 (2014-18)", "C24", "b")]
rows, prs, CN = [], [], {}
for mn, kw, tag in [("LR", {}, "default"), ("HGB", {}, "default"), ("LR", BEST["LR"], "tuned"), ("HGB", BEST["HGB"], "tuned")]:
    for k, nm, tk, src in SETUPS:
        tr, va = (a_tr, a_va) if src == "a" else (b_tr, b_va)
        c, auc, _, _ = run(mn, kw, tr, va, tk); CN[(mn, tag, k)] = c
        rows.append(dict(model=mn, settings=tag, setup=k, description=nm, n_test=len(TEST[tk][0]), pos_rate=round(TEST[tk][0].y.mean(), 4),
                         TSS=round(tss_c(c), 3), CI95=ci(boot(c)), AUC=round(auc, 3)))
    for a_, b_, what in [("B", "C", "cross-cycle minus in-cycle on C25 test (B - C)"), ("D", "A", "reverse minus in-cycle on C24 test (D - A)")]:
        d, bd = boot_paired(CN[(mn, tag, a_)], CN[(mn, tag, b_)])
        prs.append(dict(model=mn, settings=tag, comparison=what, dTSS=round(d, 3), CI95=ci(bd), excludes_zero=bool(np.percentile(bd, 2.5) > 0 or np.percentile(bd, 97.5) < 0)))
E4 = save(pd.DataFrame(rows), "e4_cross_cycle")
E4p = save(pd.DataFrame(prs), "e4_cross_cycle_paired")

# size-matched: Cycle 24 training regions subsampled to the size of the Cycle 25 training pool (5 seeds)
sm = []
target = len(b_tr)
for seed in range(5):
    regs = a_tr.ar.unique(); rs = np.random.RandomState(100 + seed); rs.shuffle(regs)
    n_cum = a_tr.groupby("ar").size().loc[regs].cumsum(); pick = set(n_cum.index[n_cum <= target])
    a_sub = a_tr[a_tr.ar.isin(pick)]
    for mn in ["LR", "HGB"]:
        c, auc, _, _ = run(mn, {}, a_sub, a_va, "C25")
        d, bd = boot_paired(c, CN[(mn, "default", "C")])
        sm.append(dict(model=mn, seed=seed, train_windows=len(a_sub), train_regions=a_sub.ar.nunique(), train_pos_rate=round(a_sub.y.mean(), 4),
                       C25_train_windows=len(b_tr), TSS_Bmatched=round(tss_c(c), 3), dTSS_vs_C=round(d, 3), CI95=ci(bd)))
E4s = save(pd.DataFrame(sm), "e4_size_matched", "Cycle 24 training subsampled to the Cycle 25 training size, scored on the same C25 test windows")

C24 train/val/test 97071 24662 91709 | C25 train/val/test 50776 12276 97866



--> phase3_out/v1/e4_cross_cycle.csv   [23.6 min]
model settings setup                       description  n_test  pos_rate   TSS           CI95   AUC
   LR  default     A in-cycle C24 (2010-13 -> 2014-18)   91709    0.0411 0.721 [0.653, 0.773] 0.914
   LR  default     B        cross C24 -> C25 (2023-24)   97866    0.1113 0.656 [0.620, 0.689] 0.914
   LR  default     C in-cycle C25 (2021-22 -> 2023-24)   97866    0.1113 0.645 [0.608, 0.677] 0.901
   LR  default     D      reverse C25 -> C24 (2014-18)   91709    0.0411 0.701 [0.643, 0.749] 0.916
  HGB  default     A in-cycle C24 (2010-13 -> 2014-18)   91709    0.0411 0.689 [0.622, 0.737] 0.919
  HGB  default     B        cross C24 -> C25 (2023-24)   97866    0.1113 0.608 [0.570, 0.642] 0.885
  HGB  default     C in-cycle C25 (2021-22 -> 2023-24)   97866    0.1113 0.549 [0.514, 0.581] 0.876
  HGB  default     D      reverse C25 -> C24 (2014-18)   91709    0.0411 0.619 [0.553, 0.668] 0.901
   LR    tuned     A in-cycle C24 (2010-13 -> 201


--> phase3_out/v1/e4_cross_cycle_paired.csv   [23.6 min]
model settings                                     comparison   dTSS             CI95  excludes_zero
   LR  default cross-cycle minus in-cycle on C25 test (B - C)  0.011  [-0.008, 0.032]          False
   LR  default     reverse minus in-cycle on C24 test (D - A) -0.020  [-0.050, 0.013]          False
  HGB  default cross-cycle minus in-cycle on C25 test (B - C)  0.059   [0.033, 0.083]           True
  HGB  default     reverse minus in-cycle on C24 test (D - A) -0.069 [-0.102, -0.034]           True
   LR    tuned cross-cycle minus in-cycle on C25 test (B - C)  0.052   [0.035, 0.070]           True
   LR    tuned     reverse minus in-cycle on C24 test (D - A) -0.021  [-0.043, 0.008]          False
  HGB    tuned cross-cycle minus in-cycle on C25 test (B - C)  0.003  [-0.013, 0.022]          False
  HGB    tuned     reverse minus in-cycle on C24 test (D - A) -0.010  [-0.042, 0.017]          False



--> phase3_out/v1/e4_size_matched.csv  (Cycle 24 training subsampled to the Cycle 25 training size, scored on the same C25 test windows)   [24.9 min]
model  seed  train_windows  train_regions  train_pos_rate  C25_train_windows  TSS_Bmatched  dTSS_vs_C            CI95
   LR     0          50524            291          0.0397              50776         0.649      0.004 [-0.015, 0.024]
  HGB     0          50524            291          0.0397              50776         0.616      0.067  [0.041, 0.092]
   LR     1          50628            293          0.0337              50776         0.645      0.001 [-0.021, 0.021]
  HGB     1          50628            293          0.0337              50776         0.582      0.033  [0.006, 0.057]
   LR     2          50621            287          0.0381              50776         0.643     -0.001 [-0.024, 0.019]
  HGB     2          50621            287          0.0381              50776         0.583      0.034  [0.007, 0.059]
   LR     3          50

## E5 — Per-year skill in Cycle 25 and calibration (models trained on Cycle 24 only; v1 Sections 4.4–4.5)

In [7]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))).reshape(-1, 1)
def ece(p, yy, k=10):
    q = pd.qcut(p, k, duplicates="drop"); d = pd.DataFrame({"p": p, "y": yy, "q": q}).groupby("q", observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((d.n * (d.p - d.y).abs()).sum() / d.n.sum())
clim24, clim25 = a_tr.y.mean(), b_tr0.y.mean()
yr_rows, cal_rows = [], []
for mn in ["LR", "HGB"]:
    m = mk(mn).fit(a_tr[FE].values, a_tr.y.values)
    pv = m.predict_proba(a_va[FE].values)[:, 1]; thr = best_t(pv, a_va.y.values)
    for yr in YEARS:
        D = B[B.time.dt.year == yr]; p = m.predict_proba(D[FE].values)[:, 1]; yy = D.y.values
        cc, n_ = pd.factorize(D.ar); c = counts(p, yy, cc, len(n_), thr)
        yr_rows.append(dict(model=mn, year=yr, windows=len(D), pos_rate=round(yy.mean(), 4), TSS_fixed=round(tss_c(c), 3), CI95=ci(boot(c)),
                            TSS_oracle=round(tss_pt(p, yy, best_t(p, yy)), 3), AUC=round(roc_auc_score(yy, p), 3) if 0 < yy.sum() < len(yy) else np.nan))
    pt = m.predict_proba(b_te[FE].values)[:, 1]; yt = b_te.y.values
    pl24 = LogisticRegression(C=1e6).fit(lg(pv), a_va.y.values)
    pb = m.predict_proba(b_va[FE].values)[:, 1]; pl25 = LogisticRegression(C=1e6).fit(lg(pb), b_va.y.values)
    r24 = brier_score_loss(yt, np.full(len(yt), clim24)); r25 = brier_score_loss(yt, np.full(len(yt), clim25))
    for nm, pp in [("raw", pt), ("Platt, Cycle 24", pl24.predict_proba(lg(pt))[:, 1]), ("recalibrated, Cycle 25 early", pl25.predict_proba(lg(pt))[:, 1])]:
        b = brier_score_loss(yt, pp)
        cal_rows.append(dict(model=mn, version=nm, Brier=round(b, 4), BSS_vs_C24_clim=round(1 - b / r24, 3), BSS_vs_C25early_clim=round(1 - b / r25, 3),
                             ECE=round(ece(pp, yt), 3), mean_pred=round(pp.mean(), 3), obs_rate=round(yt.mean(), 3)))
E5y = save(pd.DataFrame(yr_rows), "e5_per_year")
E5c = save(pd.DataFrame(cal_rows), "e5_calibration", f"climatology: C24 train {clim24:.4f}, C25 2021-22 pool {clim25:.4f}")


--> phase3_out/v1/e5_per_year.csv   [25.3 min]
model  year  windows  pos_rate  TSS_fixed           CI95  TSS_oracle   AUC
   LR  2021    17182    0.0180      0.606 [0.412, 0.787]       0.646 0.893
   LR  2022    37385    0.0480      0.680 [0.601, 0.740]       0.692 0.915
   LR  2023    50258    0.0761      0.634 [0.580, 0.687]       0.649 0.896
   LR  2024    49491    0.1482      0.664 [0.618, 0.708]       0.685 0.917
  HGB  2021    17182    0.0180      0.596 [0.400, 0.782]       0.653 0.878
  HGB  2022    37385    0.0480      0.642 [0.569, 0.702]       0.651 0.898
  HGB  2023    50258    0.0761      0.564 [0.502, 0.625]       0.576 0.870
  HGB  2024    49491    0.1482      0.628 [0.581, 0.673]       0.629 0.889



--> phase3_out/v1/e5_calibration.csv  (climatology: C24 train 0.0337, C25 2021-22 pool 0.0337)   [25.3 min]
model                      version  Brier  BSS_vs_C24_clim  BSS_vs_C25early_clim   ECE  mean_pred  obs_rate
   LR                          raw 0.1512           -0.440                -0.440 0.208      0.320     0.111
   LR              Platt, Cycle 24 0.0683            0.350                 0.350 0.024      0.088     0.111
   LR recalibrated, Cycle 25 early 0.0694            0.339                 0.339 0.036      0.105     0.111
  HGB                          raw 0.0864            0.177                 0.177 0.058      0.130     0.111
  HGB              Platt, Cycle 24 0.0804            0.234                 0.234 0.053      0.059     0.111
  HGB recalibrated, Cycle 25 early 0.0771            0.266                 0.266 0.036      0.076     0.111


## Reproduction check (label `v1` only): compare point estimates with the published CSVs

In [8]:
if LABEL == "v1" and C25_TABLE == "c25_samples_v2.pkl" and PUBLISHED and os.path.exists(f"{PUBLISHED}/leakage_audit_c24p.csv"):
    rep = []
    P1 = pd.read_csv(f"{PUBLISHED}/leakage_audit_c24p.csv")
    for _, r in P1.iterrows():
        n = E1[(E1.model == r.model) & (E1.split == r.split)].TSS
        rep.append(dict(table="leakage default", item=f"{r.model} | {r.split}", published=r.TSS, rerun=float(n.iloc[0]) if len(n) else np.nan))
    P2 = pd.read_csv(f"{PUBLISHED}/cycle_transfer_results.csv")
    for _, r in P2.iterrows():
        n = E4[(E4.model == r.model) & (E4.settings == "default") & (E4.setup == r.setup.strip()[0])].TSS
        rep.append(dict(table="cross-cycle", item=f"{r.model} | {r.setup}", published=r.TSS, rerun=float(n.iloc[0]) if len(n) else np.nan))
    P3 = pd.read_csv(f"{PUBLISHED}/leakage_tuned_5fold.csv")
    for _, r in P3.iterrows():
        n = E3[E3.model == r.model]
        for col in ["TSS_random", "TSS_grouped", "inflation"]:
            rep.append(dict(table="leakage tuned", item=f"{r.model} | {col}", published=r[col], rerun=float(n[col].iloc[0]) if len(n) else np.nan))
    R = pd.DataFrame(rep); R["diff"] = (R.rerun - R.published).round(3)
    save(R, "reproduction_check", "differences come from scikit-learn version, the extended LR grid (tuned rows) and nothing else")
else:
    print("reproduction check skipped (label", LABEL, ")")


--> phase3_out/v1/reproduction_check.csv  (differences come from scikit-learn version, the extended LR grid (tuned rows) and nothing else)   [25.3 min]
          table                                                         item  published  rerun   diff
leakage default                        1-feat (USFLUX) | random rows (leaky)      0.636  0.636  0.000
leakage default                              1-feat (USFLUX) | grouped by AR      0.629  0.633  0.004
leakage default                 1-feat (USFLUX) | forward time + AR-disjoint      0.647  0.647  0.000
leakage default        3-feat (USFLUX,TOTUSJH,R_VALUE) | random rows (leaky)      0.683  0.683  0.000
leakage default              3-feat (USFLUX,TOTUSJH,R_VALUE) | grouped by AR      0.678  0.654 -0.024
leakage default 3-feat (USFLUX,TOTUSJH,R_VALUE) | forward time + AR-disjoint      0.690  0.690  0.000
leakage default                                     LR | random rows (leaky)      0.746  0.747  0.001
leakage default                

## Figures (drawn only from the CSV files above)

In [9]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
BLUE, ORANGE, INK, INK2, GRID_ = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False})
def pci(s): lo, hi = [float(x) for x in str(s).strip("[]").split(",")]; return lo, hi
def style(ax): ax.grid(axis="y", color=GRID_, lw=0.8); ax.set_axisbelow(True)
def savefig(fig, name): fig.savefig(f"{OUT}/{name}.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.close(fig)

E1 = pd.read_csv(f"{OUT}/e1_leakage_default.csv"); E3 = pd.read_csv(f"{OUT}/e3_leakage_tuned.csv")
def panel(ax, labels, rv, rci, gv, gci, title):
    x = np.arange(len(labels)); w = 0.36
    for off, v, c, colr, nm in [(-w/2 - 0.01, rv, rci, BLUE, "Random row split (leaky)"), (w/2 + 0.01, gv, gci, ORANGE, "Grouped by active region")]:
        lo = [a - pci(b)[0] for a, b in zip(v, c)]; hi = [pci(b)[1] - a for a, b in zip(v, c)]
        ax.bar(x + off, v, w, color=colr, edgecolor="white", linewidth=2, label=nm, yerr=[lo, hi], error_kw=dict(ecolor=INK2, lw=1, capsize=2))
        for xi, vi, h in zip(x + off, v, hi): ax.text(xi, vi + h + 0.012, f"{vi:.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.5); ax.set_ylim(0, 1.08); ax.set_ylabel("TSS (24 h, M+)"); ax.set_title(title, fontsize=10, loc="left"); style(ax)
fig, axs = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [5, 3]})
r_ = E1[E1.split.str.startswith("random")].set_index("model").loc[MODELS]; g_ = E1[E1.split.str.startswith("grouped")].set_index("model").loc[MODELS]
panel(axs[0], [SHORT[m] for m in MODELS], r_.TSS.values, r_.CI95.values, g_.TSS.values, g_.CI95.values, "(a) Default hyperparameters")
panel(axs[1], ["LR", "HGB", "MLP"], E3.TSS_random.values, E3.CI_r.values, E3.TSS_grouped.values, E3.CI_g.values, "(b) Tuned with grouped CV")
axs[0].legend(frameon=False, fontsize=8.5, loc="upper left", ncol=2)
fig.suptitle(f"Leakage audit, Cycle 24, 5-fold, 95% cluster-bootstrap intervals over active regions (labels {LABEL}, run {RUN})", fontsize=10, x=0.01, ha="left")
savefig(fig, "fig1_leakage")

E4 = pd.read_csv(f"{OUT}/e4_cross_cycle.csv"); E4p = pd.read_csv(f"{OUT}/e4_cross_cycle_paired.csv")
labs = {"A": "In-cycle C24\n(2010-13 -> 2014-18)", "B": f"Cross-cycle\nC24 -> C25 (2023-{YEARS[-1] % 100})", "C": f"In-cycle C25\n(2021-22 -> 2023-{YEARS[-1] % 100})", "D": "Reverse\nC25 -> C24 (2014-18)"}
fig, ax = plt.subplots(figsize=(8.5, 4))
for off, mn, colr in [(-0.08, "LR", BLUE), (0.08, "HGB", ORANGE)]:
    d = E4[(E4.model == mn) & (E4.settings == "default")].set_index("setup").loc[list(labs)]
    lo = [t - pci(c)[0] for t, c in zip(d.TSS, d.CI95)]; hi = [pci(c)[1] - t for t, c in zip(d.TSS, d.CI95)]
    ax.errorbar(np.arange(4) + off, d.TSS, yerr=[lo, hi], fmt="o", color=colr, ecolor=colr, elinewidth=1.4, capsize=3, ms=7, mec="white", mew=1.5, label=mn)
lims = [pci(c) for c in E4[E4.settings == "default"].CI95]
ax.set_xticks(range(4)); ax.set_xticklabels(list(labs.values()), fontsize=8.5); ax.set_ylim(min(a for a, _ in lims) - 0.05, max(b for _, b in lims) + 0.05); ax.set_ylabel("TSS (24 h, M+)"); style(ax); ax.legend(frameon=False)
bc = E4p[(E4p.settings == "default") & E4p.comparison.str.contains("B - C")].set_index("model")
ax.set_title("Cross-cycle transfer (default settings); paired B - C: " + "; ".join(f"{m} {bc.loc[m, 'dTSS']:+.3f} {bc.loc[m, 'CI95']}" for m in ["LR", "HGB"]), fontsize=8.5, loc="left")
savefig(fig, "fig2_cross_cycle")

E5y = pd.read_csv(f"{OUT}/e5_per_year.csv")
fig, ax = plt.subplots(figsize=(7, 4))
for off, mn, colr in [(-0.04, "LR", BLUE), (0.04, "HGB", ORANGE)]:
    d = E5y[E5y.model == mn]; c = [pci(s) for s in d.CI95]
    ax.errorbar(d.year + off, d.TSS_fixed, yerr=[d.TSS_fixed - [a for a, _ in c], [b for _, b in c] - d.TSS_fixed], fmt="-o", color=colr, lw=1.6, capsize=3, ms=6, mec="white", mew=1.2, label=mn)
lims = [pci(c) for c in E5y.CI95]
ax.set_xticks(YEARS); ax.set_ylim(max(-1, min(a for a, _ in lims) - 0.05), min(1, max(b for _, b in lims) + 0.05)); ax.set_ylabel("TSS (fixed Cycle 24 threshold)"); style(ax); ax.legend(frameon=False)
ax.set_title("Cycle 25 skill by year, models trained on Cycle 24 only", fontsize=10, loc="left")
savefig(fig, "fig3_per_year")

E5c = pd.read_csv(f"{OUT}/e5_calibration.csv")
fig, ax = plt.subplots(figsize=(7.5, 4)); x = np.arange(3); w = 0.36
for off, mn, colr in [(-w/2 - 0.01, "LR", BLUE), (w/2 + 0.01, "HGB", ORANGE)]:
    v = E5c[E5c.model == mn].BSS_vs_C24_clim.values
    ax.bar(x + off, v, w, color=colr, edgecolor="white", linewidth=2, label=mn)
    for xi, vi in zip(x + off, v): ax.text(xi, vi + (0.015 if vi >= 0 else -0.04), f"{vi:.2f}", ha="center", fontsize=8.5)
ax.axhline(0, color=INK2, lw=1); ax.set_xticks(x); ax.set_xticklabels(["Raw\n(balanced weights)", "Platt fit\non Cycle 24", "Recalibrated on\nCycle 25 early"], fontsize=8.5)
ax.set_ylabel(f"Brier skill score, C25 2023-{YEARS[-1] % 100}\n(vs Cycle 24 climatology)"); style(ax); ax.legend(frameon=False)
ax.set_title("Calibration under the Cycle 25 base rate", fontsize=10, loc="left")
savefig(fig, "fig4_calibration")
print("figures saved", sorted(f for f in os.listdir(OUT) if f.endswith(".png")))

json.dump({"label": LABEL, "run": RUN, "c25_table": C25_TABLE, "c25_last_window": str(B.time.max()), "n_boot": NB, "selected_hyperparameters": {k: {kk: (list(vv) if isinstance(vv, tuple) else vv) for kk, vv in v.items()} for k, v in BEST.items()},
           "minutes": round((time.time() - T0) / 60, 1), "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__}},
          open(f"{OUT}/run_info.json", "w"), indent=1)
print("done in", round((time.time() - T0) / 60, 1), "min")

figures saved ['fig1_leakage.png', 'fig2_cross_cycle.png', 'fig3_per_year.png', 'fig4_calibration.png']
done in 25.3 min
